# SPATIAL INTELLIGENCE ANALYSIS ON TYPICAL RESIDENTIAL FLOOR PLAN

In [1]:
# Run once to install / upgrade TopologicPy
%pip install --upgrade topologicpy shapely

import importlib, sys
from importlib import metadata as importlib_metadata

for module_name in list(sys.modules):
    if module_name == "topologicpy" or module_name.startswith("topologicpy."):
        del sys.modules[module_name]

importlib.invalidate_caches()
print(f"Installed TopologicPy version: {importlib_metadata.version('topologicpy')}")


Note: you may need to restart the kernel to use updated packages.
Installed TopologicPy version: 0.9.50


## 1. Import Libraries

In [2]:
from topologicpy.Vertex     import Vertex
from topologicpy.Edge       import Edge
from topologicpy.Wire       import Wire
from topologicpy.Face       import Face
from topologicpy.Shell      import Shell
from topologicpy.Cell       import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster    import Cluster
from topologicpy.Topology   import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper     import Helper
from topologicpy.Grid       import Grid
from topologicpy.Graph      import Graph
from topologicpy.Color      import Color
from importlib import metadata as importlib_metadata

print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(f"Installed: {importlib_metadata.version('topologicpy')}")
print(f"Runtime  : {Helper.Version()}")


c:\Users\MOHA9808\Downloads\New folder\GML.26\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This tutorial requires topologicpy version 0.9.18 or newer.
Installed: 0.9.50
Runtime  : The version that you are using (0.9.50) is EQUAL TO the latest version available on PyPI.


## 2. Set Renderer
- VS Code: `"vscode"`
- Colab: `"colab"`
- Browser: `"browser"`

In [3]:
renderer = "vscode"

## 3. Utility Functions

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d    = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if key != "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d     = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d     = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value and str(value) in dicts:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)

print("Utility functions ready.")


Utility functions ready.


## 4. Paths Configuration

In [5]:
OBJ_PATH  = r"C:\Users\MOHA9808\Downloads\New folder\GML.26\assignments\final assignment -xx\assets\typical -a.obj"
BREP_PATH = r"C:\Users\MOHA9808\Downloads\New folder\GML.26\assignments\final assignment -xx\assets\resi floor_export_surface.brep"


## 5. Load OBJ Floor Plan

In [6]:
objects = Topology.ByOBJPath(OBJ_PATH)
if isinstance(objects, list):
    print(f"Loaded {len(objects)} objects from OBJ")
else:
    print("Loaded:", objects)


Loaded 2 objects from OBJ


## 6. Export to BREP

In [7]:
cluster = Cluster.ByTopologies(objects) if isinstance(objects, list) else objects
status  = Topology.ExportToBREP(cluster, path=BREP_PATH, overwrite=True)
print(f"Export status : {status}")
print(f"BREP saved to : {BREP_PATH}")


Export status : True
BREP saved to : C:\Users\MOHA9808\Downloads\New folder\GML.26\assignments\final assignment -xx\assets\resi floor_export_surface.brep


## 7. Mesh Edge Removal — Dissolve Triangulation

In [8]:
from shapely.geometry import Polygon as ShapelyPolygon, MultiPolygon
from shapely.ops      import unary_union

# ── helpers ──────────────────────────────────────────────────────────────────
def shapely_to_face(polygon):
    coords = list(polygon.exterior.coords)[:-1]
    if len(coords) < 3:
        return None
    outer = Wire.ByVertices(
        [Vertex.ByCoordinates(x, y, 0.0) for x, y in coords], close=True)
    if outer is None:
        return None
    holes = []
    for interior in polygon.interiors:
        hc = list(interior.coords)[:-1]
        if len(hc) < 3:
            continue
        hw = Wire.ByVertices(
            [Vertex.ByCoordinates(x, y, 0.0) for x, y in hc], close=True)
        if hw is not None:
            holes.append(hw)
    return Face.ByWires(outer, holes) if holes else Face.ByWire(outer)

def extract_polygons(geom):
    """Recursively extract only Polygon types — drops lines/points."""
    if geom.geom_type == "Polygon":
        return [geom] if geom.area > 0 else []
    elif geom.geom_type in ("MultiPolygon", "GeometryCollection"):
        result = []
        for g in geom.geoms:
            result.extend(extract_polygons(g))
        return result
    return []

# ── load faces directly from OBJ (skip BREP for dissolve step) ───────────────
objects_raw = Topology.ByOBJPath(OBJ_PATH)
cluster_raw = Cluster.ByTopologies(objects_raw) if isinstance(objects_raw, list) else objects_raw
tri_faces   = Topology.Faces(cluster_raw)
print(f"Faces from OBJ: {len(tri_faces)}")

# ── build valid Shapely polygons ──────────────────────────────────────────────
shapely_polys = []
for f in tri_faces:
    verts  = Topology.Vertices(f)
    coords = [(v.X(), v.Y()) for v in verts]
    unique = list(set(coords))
    if len(unique) < 3:
        continue
    poly = ShapelyPolygon(coords)
    if not poly.is_valid:
        poly = poly.buffer(0)
    if poly.is_valid and not poly.is_empty:
        shapely_polys.append(poly)

print(f"Valid Shapely polygons: {len(shapely_polys)}")

# ── dissolve ──────────────────────────────────────────────────────────────────
dissolved = unary_union(shapely_polys)
if not dissolved.is_valid:
    dissolved = dissolved.buffer(0)
print(f"Dissolved type: {dissolved.geom_type}")

# ── convert back to TopologicPy Face ─────────────────────────────────────────
poly_list = extract_polygons(dissolved)
print(f"Extracted polygons: {len(poly_list)}")

if len(poly_list) == 0:
    print("ERROR: No valid polygons — check OBJ path and geometry.")
elif len(poly_list) == 1:
    raw     = shapely_to_face(poly_list[0])
    gallery = Topology.RemoveCollinearEdges(raw) or raw
    print("Gallery (single face):", gallery)
else:
    parts   = [shapely_to_face(p) for p in poly_list]
    parts   = [f for f in parts if f is not None]
    parts   = [(Topology.RemoveCollinearEdges(f) or f) for f in parts]
    gallery = Cluster.ByTopologies(parts) if len(parts) > 1 else parts[0]
    print(f"Gallery ({len(parts)} parts):", gallery)


Faces from OBJ: 1238
Valid Shapely polygons: 1238
Dissolved type: Polygon
Extracted polygons: 1
Gallery (single face): <topologic_core.Face object at 0x0000018F5A424D30>


In [9]:
# ── DEBUG: inspect actual face coordinates ──────────────────────────────────
objects_raw = Topology.ByOBJPath(OBJ_PATH)
cluster_raw = Cluster.ByTopologies(objects_raw) if isinstance(objects_raw, list) else objects_raw
tri_faces   = Topology.Faces(cluster_raw)

print(f"Total faces: {len(tri_faces)}")
print("\n--- First 5 faces ---")
for i, f in enumerate(tri_faces[:5]):
    verts  = Topology.Vertices(f)
    coords = [(round(v.X(),4), round(v.Y(),4), round(v.Z(),4)) for v in verts]
    print(f"Face {i}: {len(verts)} verts → {coords}")

Total faces: 1238

--- First 5 faces ---
Face 0: 3 verts → [(-29.8025, -1.3754, 0.2), (-29.8025, 3.2564, 0.2), (-27.7545, -0.7754, 0.2)]
Face 1: 3 verts → [(-26.5545, 3.2564, 0.2), (-27.6545, -0.7754, 0.2), (-27.7545, -0.7754, 0.2)]
Face 2: 3 verts → [(-26.5545, 3.2564, 0.2), (-26.5545, -0.9504, 0.2), (-27.6545, -0.7754, 0.2)]
Face 3: 3 verts → [(-29.8025, 3.2564, 0.2), (-26.5545, 3.2564, 0.2), (-27.7545, -0.7754, 0.2)]
Face 4: 3 verts → [(-27.6545, -1.9754, 0.2), (-27.6545, -0.7754, 0.2), (-26.5545, -0.9504, 0.2)]


## 8. Show Floor Plan Geometry

In [10]:
Topology.Show(gallery,
              camera          = [0, 0, 6],
              faceColor       = [210, 210, 250],
              faceOpacity     = 1,
              edgeColor       = "black",
              edgeWidth       = 3,
              showVertices    = False,
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)


## 9. Create 1.5m Grid Overlay

In [10]:
b_r    = Wire.BoundingRectangle(gallery)
b_face = Face.ByWire(b_r)
d      = Topology.Dictionary(b_r)
xmin   = Dictionary.ValueAtKey(d, "xmin")
xmax   = Dictionary.ValueAtKey(d, "xmax")
ymin   = Dictionary.ValueAtKey(d, "ymin")
ymax   = Dictionary.ValueAtKey(d, "ymax")
width  = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

if b_face is None or width is None or length is None:
    raise ValueError("Bounding rectangle face failed. Check the floor-plan geometry before creating the grid.")

print(f"Bounding box: X={xmin:.2f}→{xmax:.2f}  Y={ymin:.2f}→{ymax:.2f}")
print(f"Width={width:.2f}  Length={length:.2f}")

grid_cell = 1
uRange    = [i * grid_cell for i in range(int(width  / grid_cell) + 2)]
vRange    = [i * grid_cell for i in range(int(length / grid_cell) + 2)]

grid = Grid.EdgesByDistances(b_face, clip=False, uRange=uRange, vRange=vRange)
if grid is None:
    raise ValueError("Grid creation failed. TopologicPy did not return a valid edge grid.")

print("Grid created:", grid)


Bounding box: X=-29.80→34.31  Y=-29.35→7.71
Width=64.11  Length=37.07
Grid created: <topologic_core.Cluster object at 0x0000018F3C7811B0>


## 10. Show Floor Plan + Grid

In [11]:
Topology.Show(gallery, grid,
              camera          = [0, 0, 6],
              faceColor       = [210, 210, 250],
              faceOpacity     = 1,
              edgeColor       = "black",
              edgeWidth       = 3,
              showVertices    = False,
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)


## 11. Slice Floor Plan with Grid → Shell

In [12]:
shell = Topology.Slice(gallery, grid)
if shell is None:
    raise ValueError("Slicing the floor plan with the grid failed.")

faces = Topology.Faces(shell)
if not faces:
    raise ValueError("Slicing did not produce any faces.")

print(f"Shell grid cells: {len(faces)}")

for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)

print("Face IDs assigned.")


Shell grid cells: 2083
Face IDs assigned.


## 12. Show Shell

In [13]:
Topology.Show(shell,
              camera          = [0, 0, 6],
              faceColor       = [210, 210, 250],
              faceOpacity     = 0.9,
              edgeColor       = "black",
              edgeWidth       = 3,
              showVertices    = False,
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)


## 13. Build Navigation & Analysis Graphs

In [14]:
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph   = Graph.ByTopology(shell)
g_verts          = Graph.Vertices(analysis_graph)
print(f"Analysis graph nodes : {len(g_verts)}")
print(f"Navigation graph     : {navigation_graph}")


Analysis graph nodes : 2082
Navigation graph     : <topologic_core.Graph object at 0x0000018F43106B70>


## 14. Show Analysis Graph

In [15]:
Topology.Show(analysis_graph,
              camera          = [0, 0, 6],
              vertexSize      = 4,
              vertexColor     = "red",
              edgeColor       = "black",
              backgroundColor = "black",
              width           = 800,
              height          = 600,
              renderer        = renderer)


## 15a. Minimum Spanning Tree (Demo on Simple Graph)

In [16]:
import time

cc1 = CellComplex.Prism()
cc2 = Topology.Translate(cc1, 1.1, 0, 0)
cc3 = Topology.Translate(cc2, 1.1, 0, 0)
g1  = Graph.ByTopology(cc1)
g2  = Graph.ByTopology(cc2)
g3  = Graph.ByTopology(cc3)
g2  = Graph.MinimumSpanningTree(g2)

Topology.Show(g1, g2, g3,
              vertexSize      = 12,
              vertexColor     = "red",
              edgeColor       = "lightgrey",
              edgeWidth       = 4,
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)

print("Density  g1/g2/g3:", Graph.Density(g1), Graph.Density(g2), Graph.Density(g3))
print("Diameter g1/g2/g3:", Graph.Diameter(g1), Graph.Diameter(g2), Graph.Diameter(g3))


Density  g1/g2/g3: 0.42857142857142855 0.25 0.42857142857142855
Diameter g1/g2/g3: 3 5 3


## 15b. Shortest Path

In [17]:
import time

start_vertex  = Vertex.ByCoordinates(xmin + 2, ymax - 2, 0)
end_vertex    = Vertex.ByCoordinates(xmax - 2, ymin + 2, 0)
crg           = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)

t0            = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
print(f"Shortest path computed in {round(time.time()-t0, 2)}s")

t0            = time.time()
straight_path = Wire.Straighten(shortest_path, host=gallery)
print(f"Straighten computed in    {round(time.time()-t0, 2)}s")

print(f"Original path length  : {round(Wire.Length(shortest_path), 2)}")
print(f"Straightened length   : {round(Wire.Length(straight_path), 2)}")

for edge in Topology.Edges(shortest_path):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(["width","color"], [7,"red"]))
for edge in Topology.Edges(straight_path):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(["width","color"], [7,"blue"]))

Topology.Show(gallery, shortest_path, straight_path,
              camera          = [0, 0, 6],
              faceColor       = [210, 210, 250],
              faceOpacity     = 1,
              edgeColorKey    = "color",
              edgeWidthKey    = "width",
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)


Shortest path computed in 1.37s
Straighten computed in    46.53s
Original path length  : 85.39
Straightened length   : 75.71


## 16a. Degree Centrality (Connectivity)
Number of direct connections per node — predicts local movement intensity.

In [18]:
centrality_list = Graph.DegreeCentrality(analysis_graph, colorScale="thermal")
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_     = transfer_dicts_by_key(faces, g_verts, "face_id")

Topology.Show(faces,
              faceColorKey    = "dc_color",
              faceOpacity     = 1,
              showEdges       = False,
              showVertices    = False,
              camera          = [0, 0, 6],
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)


## 16b. Closeness Centrality (Integration)
How close each node is to all others — global spatial accessibility.

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_     = transfer_dicts_by_key(faces, g_verts, "face_id")

Topology.Show(faces,
              faceColorKey    = "cc_color",
              faceOpacity     = 1,
              showEdges       = False,
              showVertices    = False,
              camera          = [0, 0, 6],
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)


## 16c. Betweenness Centrality (Choice)
How often each node lies on shortest paths between other nodes.

In [ ]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_     = transfer_dicts_by_key(faces, g_verts, "face_id")

Topology.Show(faces,
              faceColorKey    = "bc_color",
              faceOpacity     = 1,
              showEdges       = False,
              showVertices    = False,
              camera          = [0, 0, 6],
              backgroundColor = "white",
              width           = 800,
              height          = 600,
              renderer        = renderer)
